# 03: DMD vs EDMD on the Lorenz System

Goal: compare Exact DMD and EDMD on a nonlinear chaotic dynamical system.

Main idea:

- DMD fits linear evolution in the original state space: `x(k+1) ≈ A x(k)`.
- EDMD first lifts the state into hand-built dictionary features: `Psi(x)`.
- Then EDMD fits linear evolution in the lifted observable space: `Psi(x(k+1)) ≈ K Psi(x(k))`.


## 1. Import packages and local modules

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dmd import exact_dmd, make_lorenz, reconstruct, relative_error, snapshot_pair
from edmd import PolynomialDictionary, edmd_one_step, edmd_rollout, fit_edmd

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)


## 2. Generate Lorenz data

Lorenz is useful here because it is low-dimensional but strongly nonlinear and chaotic. This makes it a good stress test for DMD.

In [ ]:
dt = 0.01
t, data = make_lorenz(nt=2500, dt=dt, x0=np.array([1.0, 1.0, 1.0]))

# Drop the initial transient so the comparison focuses on the attractor.
burn_in = 500
t = t[burn_in:] - t[burn_in]
data = data[:, burn_in:]

print("data shape:", data.shape)


In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
ax.plot(data[0], data[1], data[2], linewidth=0.8)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Lorenz trajectory")
plt.tight_layout()


## 3. Train/test split

For chaotic systems, long-term pointwise prediction is not expected to remain accurate forever. A fair comparison is:

- one-step prediction error: how well the learned map predicts the next state;
- rollout error: how fast errors accumulate when the model is iterated forward.

In [ ]:
train_steps = 1200
test_steps = 500

train_data = data[:, :train_steps]
test_data = data[:, train_steps:train_steps + test_steps]

X_train, Y_train = snapshot_pair(train_data)
print("X_train:", X_train.shape, "Y_train:", Y_train.shape)


## 4. Fit Exact DMD

DMD uses only the original variables `[x, y, z]`. It is therefore a linear model in the original state space.

In [ ]:
dmd_model = exact_dmd(X_train, Y_train, rank=3)

dmd_train_rollout = reconstruct(dmd_model, train_steps, x0=train_data[:, 0])
dmd_test_rollout = reconstruct(dmd_model, test_steps, x0=test_data[:, 0])

# One-step DMD prediction: x(k+1) = Phi Lambda Phi^+ x(k)
dmd_one_step_matrix = dmd_model.modes @ np.diag(dmd_model.eigvals) @ np.linalg.pinv(dmd_model.modes)
dmd_test_one_step = (dmd_one_step_matrix @ test_data[:, :-1]).real

print("DMD train rollout error:", relative_error(train_data, dmd_train_rollout))
print("DMD test rollout error:", relative_error(test_data, dmd_test_rollout))
print("DMD test one-step error:", relative_error(test_data[:, 1:], dmd_test_one_step))


## 5. Fit EDMD with a hand-built polynomial dictionary

For Lorenz, a simple first attempt is a polynomial dictionary:

`[1, x, y, z, x^2, xy, xz, y^2, yz, z^2]`

This is not the best possible dictionary, but it is interpretable and directly shows how EDMD extends DMD.

In [ ]:
dictionary = PolynomialDictionary(n_states=3, degree=2, include_bias=True)
print("number of dictionary terms:", dictionary.n_features)
print(dictionary.names(["x", "y", "z"]))

edmd_model = fit_edmd(X_train, Y_train, dictionary, rcond=1e-8)

edmd_train_rollout = edmd_rollout(edmd_model, train_data[:, 0], train_steps)
edmd_test_rollout = edmd_rollout(edmd_model, test_data[:, 0], test_steps)
edmd_test_one_step = edmd_one_step(edmd_model, test_data[:, :-1])

print("EDMD train rollout error:", relative_error(train_data, edmd_train_rollout))
print("EDMD test rollout error:", relative_error(test_data, edmd_test_rollout))
print("EDMD test one-step error:", relative_error(test_data[:, 1:], edmd_test_one_step))


## 6. Compare one-step predictions

One-step prediction is the cleanest way to show that EDMD has learned a better nonlinear local map than DMD.

In [ ]:
labels = ["x", "y", "z"]
window = 250

plt.figure(figsize=(10, 7))
for i, label in enumerate(labels):
    plt.subplot(3, 1, i + 1)
    plt.plot(t[train_steps + 1:train_steps + 1 + window], test_data[i, 1:1 + window], label="true", linewidth=2)
    plt.plot(t[train_steps + 1:train_steps + 1 + window], dmd_test_one_step[i, :window], "--", label="DMD")
    plt.plot(t[train_steps + 1:train_steps + 1 + window], edmd_test_one_step[i, :window], ":", label="EDMD", linewidth=2)
    plt.ylabel(label)
    if i == 0:
        plt.legend(ncol=3)
plt.xlabel("time")
plt.suptitle("One-step prediction on Lorenz test data")
plt.tight_layout()


## 7. Compare rollout predictions

Rollout is harder because prediction errors are fed back into the model. For chaotic systems, both methods eventually diverge from the exact trajectory. The useful question is which method stays close for longer.

In [ ]:
plt.figure(figsize=(10, 7))
for i, label in enumerate(labels):
    plt.subplot(3, 1, i + 1)
    plt.plot(t[train_steps:train_steps + window], test_data[i, :window], label="true", linewidth=2)
    plt.plot(t[train_steps:train_steps + window], dmd_test_rollout[i, :window], "--", label="DMD")
    plt.plot(t[train_steps:train_steps + window], edmd_test_rollout[i, :window], ":", label="EDMD", linewidth=2)
    plt.ylabel(label)
    if i == 0:
        plt.legend(ncol=3)
plt.xlabel("time")
plt.suptitle("Autonomous rollout on Lorenz test data")
plt.tight_layout()


## 8. Error accumulation

This plot is useful for your research notes: EDMD can improve the local map, but chaos still makes long-term trajectory matching difficult.

In [ ]:
dmd_error_t = np.linalg.norm(test_data - dmd_test_rollout, axis=0) / np.linalg.norm(test_data, axis=0)
edmd_error_t = np.linalg.norm(test_data - edmd_test_rollout, axis=0) / np.linalg.norm(test_data, axis=0)

plt.figure(figsize=(7, 4))
plt.semilogy(t[train_steps:train_steps + test_steps], dmd_error_t, label="DMD rollout")
plt.semilogy(t[train_steps:train_steps + test_steps], edmd_error_t, label="EDMD rollout")
plt.xlabel("time")
plt.ylabel("relative state error")
plt.title("Error accumulation during autonomous rollout")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()


## 9. Conclusion for PhD-application narrative

- DMD is the baseline: it assumes linear evolution in the original state variables.
- EDMD changes the representation: it lifts the state into a dictionary of observables.
- On Lorenz, polynomial EDMD should improve one-step nonlinear prediction compared with DMD.
- Long rollout is still hard because Lorenz is chaotic and EDMD does not guarantee globally stable lifted dynamics.
- This naturally motivates research questions about dictionary learning, physics-informed observables, and Koopman learning for nonlinear PDE/fluid systems.